# Distributed training: DataParallel, DistributedDataParallel (DDP), and sharding

_PyTorch_

**Replicate the model on every GPU, split the batch, and all-reduce the gradients so all copies stay in sync — DDP is the standard way.**

Data parallelism in one sentence: put a full copy of the model on every GPU, give each copy a
       different slice of the batch, let each compute gradients on its slice, then average the gradients across all
       copies so every model stays identical.

       That averaging step is an all-reduce: a collective operation where every GPU contributes its gradient
       tensor, they are summed (then divided by N), and the same averaged result is handed back to every GPU. Because
       all copies start equal and apply the same averaged gradient, they take the same optimizer step and remain
       byte-for-byte identical forever. It is exactly the &ldquo;map then combine&rdquo; shape of a Spark job
       (spark-intro), specialized to gradients.

---

This notebook is a practice scaffold for the **Distributed training: DataParallel, DistributedDataParallel (DDP), and sharding** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
# ddp_train.py  —  launch with:
#   torchrun --standalone --nproc_per_node=4 ddp_train.py
# (one process per GPU; torchrun sets RANK / WORLD_SIZE / LOCAL_RANK)
import os
import torch
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist


def main():
    # --- 1. Join the process group -------------------------------------
    # 'nccl' is the fast GPU backend. torchrun provides the env vars.
    dist.init_process_group(backend="nccl")
    rank        = dist.get_rank()              # global process index (0..N-1)
    world_size  = dist.get_world_size()        # total number of GPUs (N)
    local_rank  = int(os.environ["LOCAL_RANK"])# GPU index on this machine
    torch.cuda.set_device(local_rank)          # pin this process to its GPU
    device = torch.device("cuda", local_rank)

    torch.manual_seed(0)                       # same seed -> identical init

    # --- 2. A tiny synthetic dataset -----------------------------------
    X = torch.randn(8000, 784)
    y = torch.randint(0, 10, (8000,))
    dataset = TensorDataset(X, y)

    # DistributedSampler hands each rank a DISJOINT shard of the data.
    # Without it, every GPU would see the SAME batches (a classic bug).
    sampler = DistributedSampler(dataset, num_replicas=world_size,
                                 rank=rank, shuffle=True)
    local_batch = 32
    loader = DataLoader(dataset, batch_size=local_batch, sampler=sampler,
                        drop_last=True)        # drop_last keeps step counts
                                               # equal -> no all-reduce deadlock
    eff_batch = local_batch * world_size       # effective batch size
    if rank == 0:
        print(f"world_size={world_size}  local_batch={local_batch}  "
              f"effective_batch={eff_batch}")

    # --- 3. Model, wrapped in DDP --------------------------------------
    model = nn.Sequential(
        nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)
    ).to(device)
    model = DDP(model, device_ids=[local_rank])   # broadcasts weights from
                                                  # rank 0; averages grads via
                                                  # all-reduce in backward()

    # Linear scaling rule: scale base lr by the number of GPUs.
    base_lr = 0.05
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=base_lr * world_size, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()            # expects raw logits + class idx

    # --- 4. The per-rank training loop ---------------------------------
    for epoch in range(5):
        sampler.set_epoch(epoch)               # reshuffle differently each epoch
        model.train()
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)                 # forward on this rank's shard
            loss = loss_fn(logits, yb)
            loss.backward()                    # DDP all-reduces grads HERE
            optimizer.step()                   # every rank takes the same step
            running += loss.item()
        # Only rank 0 logs (otherwise N copies of every line).
        if rank == 0:
            print(f"epoch {epoch}  loss {running / len(loader):.4f}")

    # --- 5. Save ONCE, from rank 0 only --------------------------------
    if rank == 0:
        # .module unwraps the DDP wrapper to the plain model's state_dict
        torch.save(model.module.state_dict(), "model.pt")
        print("saved checkpoint from rank 0")

    dist.destroy_process_group()               # clean shutdown


if __name__ == "__main__":
    main()

## Visualize the data & results

_How much faster does DDP train as you add GPUs? Throughput (images/sec) and speedup vs the number of GPUs under a simple Amdahl-style model with communication overhead — near-linear at first, then sub-linear as all-reduce cost grows._

In [ ]:
import numpy as np

# Amdahl-style per-epoch time model with all-reduce communication overhead.
#   t(N) = t1 * (s + (1-s)/N)  +  c*(N-1)
# s = serial (non-parallelizable) fraction; c = per-step comm cost.
t1 = 1.0          # single-GPU epoch time (normalized)
s  = 0.02         # 2% of work is serial (data loading, fixed overhead)
c  = 0.0008 * t1  # communication grows with the number of GPUs

Ns = np.array([1, 2, 4, 8, 16, 32])
t  = t1 * (s + (1 - s) / Ns) + c * (Ns - 1)

base_throughput = 1000.0                 # images/sec on 1 GPU
throughput = base_throughput * (t1 / t)  # faster epoch -> higher throughput
ideal      = base_throughput * Ns        # perfect linear scaling
speedup    = t1 / t

for n, thr, sp in zip(Ns, throughput, speedup):
    print(f"N={n:>2d}  throughput={thr:8.1f} img/s  speedup={sp:5.2f}x "
          f"(ideal {n}x)")
# N= 1  throughput=  1000.0 img/s  speedup= 1.00x (ideal 1x)
# N= 2  throughput=  1958.0 img/s  speedup= 1.96x (ideal 2x)
# N= 4  throughput=  3740.6 img/s  speedup= 3.74x (ideal 4x)
# N= 8  throughput=  6752.4 img/s  speedup= 6.75x (ideal 8x)
# N=16  throughput= 10724.2 img/s  speedup=10.72x (ideal 16x)
# N=32  throughput= 13257.6 img/s  speedup=13.26x (ideal 32x)

import matplotlib.pyplot as plt
plt.plot(Ns, ideal, "--", color="#8b949e", label="ideal (linear)")
plt.plot(Ns, throughput, "o-", color="#4ea1ff", label="DDP (modeled)")
plt.xlabel("number of GPUs (N)")
plt.ylabel("throughput (images/sec)")
plt.title("DDP throughput vs number of GPUs")
plt.legend()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Compute the effective batch size for a DDP run. Set local_batch = 32 and world_size = 8, then print local_batch * world_size. Also print the per-GPU shard size of an 80,000-image dataset (80000 // world_size). Predict both numbers before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Multiply the per-GPU batch by the number of GPUs: local_batch * world_size. — _Each GPU processes its own local batch and their gradients are averaged, so the optimizer steps on the concatenated big batch._
- Floor-divide the dataset size by world_size for the per-rank shard. — _The DistributedSampler hands each rank a disjoint slice, so each rank sees 1/N of the data per epoch._

**Answer:** local_batch = 32
world_size = 8
print(local_batch * world_size)       # 256  -- effective batch size
print(80000 // world_size)            # 10000  -- images per GPU per epoch

</details>

**Problem 2.** Type this in Colab. Build a DistributedSampler by hand (no real cluster needed). Make a TensorDataset of 12 items, then create DistributedSampler(ds, num_replicas=4, rank=0, shuffle=False) and print list(sampler) &mdash; the indices rank 0 owns. Then build the same sampler for rank=1 and print its indices. Notice they are disjoint.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Construct a sampler per rank with num_replicas=4 and a different rank. — _Each rank gets a disjoint shard; without the sampler every rank would iterate the SAME data and you'd just do the work N times._
- Print list(sampler) for rank 0 and rank 1. — _Seeing the index sets makes the disjoint-shard behavior concrete._

**Answer:** import torch
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler

ds = TensorDataset(torch.arange(12))
s0 = DistributedSampler(ds, num_replicas=4, rank=0, shuffle=False)
s1 = DistributedSampler(ds, num_replicas=4, rank=1, shuffle=False)
print(list(s0))     # [0, 4, 8]
print(list(s1))     # [1, 5, 9]
# disjoint shards -- rank 0 and rank 1 never see the same index

</details>

**Problem 3.** Type this in Colab. Demonstrate the set_epoch gotcha. Build a shuffling DistributedSampler(ds, num_replicas=2, rank=0, shuffle=True) over a 10-item dataset. Print list(sampler) twice with NO set_epoch (same order), then call sampler.set_epoch(1) and print again (new order).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print the sampler's indices twice without changing the epoch. — _A shuffling sampler reuses the same seed each epoch unless told otherwise, so the shuffle is identical &mdash; the famous bug._
- Call sampler.set_epoch(1) then print again. — _set_epoch mixes the epoch into the seed so each epoch reshuffles differently across ranks._

**Answer:** import torch
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler

ds = TensorDataset(torch.arange(10))
sampler = DistributedSampler(ds, num_replicas=2, rank=0, shuffle=True)

sampler.set_epoch(0)
print(list(sampler))     # e.g. [4, 1, 7, 5, 3]
sampler.set_epoch(0)
print(list(sampler))     # SAME order -- identical shuffle (the bug)
sampler.set_epoch(1)
print(list(sampler))     # different order -- correct reshuffle

</details>

**Problem 4.** Type this in Colab. Wrap a tiny model in nn.DataParallel (works on CPU/single GPU, unlike DDP). Build nn.Linear(4, 2), wrap it as dp = nn.DataParallel(model), run a (6, 4) input through it, and print the output shape. Then unwrap with dp.module and confirm it is the original Linear.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap the model with nn.DataParallel(model) and call it on a batch. — _DataParallel is the single-process API; it runs even with one device, so it's the one you can demo in Colab._
- Access dp.module to reach the underlying model. — _Both DataParallel and DDP store the real model under .module &mdash; you need it to save a clean state_dict._

**Answer:** import torch
import torch.nn as nn

model = nn.Linear(4, 2)
dp = nn.DataParallel(model)
x = torch.randn(6, 4)
out = dp(x)
print(out.shape)                 # torch.Size([6, 2])
print(type(dp.module).__name__)  # Linear  -- the unwrapped model
print(dp.module is model)        # True

</details>

**Problem 5.** Type this in Colab. Show the .module checkpoint gotcha. Wrap nn.Linear(4, 2) in nn.DataParallel, then print the FIRST key of dp.state_dict() (note the module. prefix) versus dp.module.state_dict() (no prefix). Save the clean one with torch.save and confirm it loads back into a plain nn.Linear(4, 2).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare list(dp.state_dict())[0] with list(dp.module.state_dict())[0]. — _Saving the wrapper's state_dict prepends module. to every key, which then fails to load into the bare model._
- Save dp.module.state_dict() and load_state_dict into a fresh nn.Linear. — _Unwrapping first gives keys that match the plain model exactly._

**Answer:** import torch
import torch.nn as nn

dp = nn.DataParallel(nn.Linear(4, 2))
print(list(dp.state_dict())[0])         # module.weight  -- prefixed!
print(list(dp.module.state_dict())[0])  # weight         -- clean

torch.save(dp.module.state_dict(), "m.pt")
plain = nn.Linear(4, 2)
plain.load_state_dict(torch.load("m.pt"))   # loads cleanly
print("loaded ok")                          # loaded ok

</details>

**Problem 6.** Type this in Colab. Initialize a single-process &ldquo;distributed&rdquo; group with the gloo CPU backend and do a real all_reduce. Set the env vars, call dist.init_process_group("gloo", rank=0, world_size=1), all-reduce a tensor [1.0, 2.0, 3.0] with ReduceOp.SUM, print it, then destroy_process_group().

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set MASTER_ADDR/MASTER_PORT and call init_process_group with world_size=1. — _gloo runs on CPU, so a 1-process group is the only DDP-family primitive you can actually execute in Colab._
- Call dist.all_reduce(t, op=ReduceOp.SUM). — _All-reduce is the collective DDP uses to average gradients; with one rank the sum just returns the same tensor, proving the call works._

**Answer:** import os, torch
import torch.distributed as dist

os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
dist.init_process_group("gloo", rank=0, world_size=1)

t = torch.tensor([1.0, 2.0, 3.0])
dist.all_reduce(t, op=dist.ReduceOp.SUM)
print(t)                       # tensor([1., 2., 3.])  (sum over 1 rank)
print(dist.get_world_size())   # 1
dist.destroy_process_group()

</details>

**Problem 7.** Type this in Colab. Apply the linear scaling rule in code. Given base_lr = 0.1 tuned at a local batch of 32, and a DDP run with world_size = 8, compute the scaled learning rate and the effective batch size, then build an SGD optimizer over nn.Linear(10, 2) with that learning rate and print optimizer.param_groups[0]["lr"].

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale the learning rate: base_lr * world_size. — _A bigger effective batch has less gradient noise, so the linear scaling rule raises the learning rate by N to keep step magnitudes right._
- Pass the scaled value into torch.optim.SGD and read it back from param_groups. — _param_groups is the source of truth for what the optimizer will actually use._

**Answer:** import torch
import torch.nn as nn

base_lr, world_size, local_batch = 0.1, 8, 32
scaled_lr = base_lr * world_size
print(scaled_lr)                       # 0.8
print(local_batch * world_size)        # 256  -- effective batch

model = nn.Linear(10, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=scaled_lr, momentum=0.9)
print(optimizer.param_groups[0]["lr"]) # 0.8

</details>

**Problem 8.** Type this in Colab. Demonstrate gradient accumulation (the single-GPU stand-in for a big effective batch). On nn.Linear(4, 1) with K = 4 micro-batches of shape (2, 4), run forward/backward 4 times, dividing each loss by K, and call optimizer.step() and zero_grad() only ONCE after the loop. Print the loss of each micro-step. Use torch.manual_seed(0).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Accumulate (loss / K).backward() across K micro-batches without stepping. — _Gradients add up across backward calls, so K micro-batches give the same averaged gradient as one batch of K&times; the size._
- Call optimizer.step() then optimizer.zero_grad() once, after the loop. — _Stepping once per K micro-batches yields an effective batch of K&times; the micro-batch with 1/K the peak memory._

**Answer:** import torch
import torch.nn as nn

torch.manual_seed(0)
model = nn.Linear(4, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
K = 4
optimizer.zero_grad()
for k in range(K):
    xb = torch.randn(2, 4)
    yb = torch.randn(2, 1)
    loss = ((model(xb) - yb) ** 2).mean() / K   # scale by K
    loss.backward()                             # grads ACCUMULATE
    print(round(loss.item(), 4))
# 0.4961
# 0.2473
# 0.6014
# 0.2105
optimizer.step()        # one step on the accumulated gradient
optimizer.zero_grad()   # clear for the next effective batch

</details>